In [ ]:
# %% tags=["parameters"]
# 這裡設定的變數可以被外部的 Papermill 覆寫
target_folder = r"Y:\科組資料夾\風控企劃科\10 專案\專案-Clearstream保管業務"
filename = "CBL_Report.html"

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import urllib
import urllib3
import shutil
import os
from sqlalchemy import create_engine
import requests
from datetime import datetime
import json

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
#  fetch FX rate data
def fetch_sinopac_rates():
    url = "https://mma.sinopac.com/ws/share/rate/ws_exchange.ashx?exchangeType=REMIT&Cross=genREMITResult"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://mma.sinopac.com/"
    }
    
    rates = {"TWD": 1.0}
    fetch_time = datetime.now().strftime("%Y-%m-%d %H:%M")
    
    try:
        res = requests.get(url, headers=headers, timeout=10, verify=False)
        res.raise_for_status()
        raw_text = res.text.strip()
        start_idx = raw_text.find("(") + 1
        end_idx = raw_text.rfind(")")
        
        if start_idx > 0 and end_idx > start_idx:
            json_str = raw_text[start_idx:end_idx]
            data = json.loads(json_str)
            
            for item in data:
                if data and "SubInfo" in data[0]:
                    exchange_list = data[0]["SubInfo"]
                    for item in exchange_list:
                        cur = item.get("DataValue4") 
                        rb_rate = item.get("DataValue2") 
                        if cur and rb_rate:
                            rates[cur.strip().upper()] = float(rb_rate)
            print(f"✅ 成功解析 {len(rates)-1} 種外幣匯率")
                
    except Exception as e:
        print(f"⚠️ 匯率處理失敗，將使用預設值。錯誤: {e}")
        rates.update({"USD": 32.0, "EUR": 34.5, "JPY": 0.22})
        fetch_time = "取得失敗 (解析錯誤)"
    return rates, fetch_time

fx_rates, fx_fetch_time = fetch_sinopac_rates()

def to_usd(row, val_col, cur_col):
    val = row[val_col]
    cur = row[cur_col]
    if pd.isna(val) or pd.isna(cur): return 0.0
    cur = str(cur).upper()
    if cur == "USD": return val
    rate_cur = fx_rates.get(cur, 1.0)
    rate_usd = fx_rates.get("USD", 1.0)
    return val * (rate_cur / rate_usd)

In [ ]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=128.110.24.133;"
    "DATABASE=MIDOFFICE;"
    "TrustServerCertificate=yes;"
    "UID=fixuser;"
    "PWD=7ujm4rfv;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# get & preprocess data from SQL (Loan Activity)
query = "SELECT * FROM HST_ASL_D"
loan_df = pd.read_sql_query(query, engine)

loan_df["Pos_date"] = pd.to_datetime(loan_df["Pos_date"], errors="coerce")
loan_df["LoanQuantity"] = pd.to_numeric(loan_df["LoanQuantity"], errors="coerce")
loan_df["AccruedCommission"] = pd.to_numeric(loan_df["AccruedCommission"], errors="coerce")
loan_df["Month"] = loan_df["Pos_date"].dt.to_period("M").astype(str)

loan_df["LoanQuantity_USD"] = loan_df.apply(lambda x: to_usd(x, "LoanQuantity", "LoanValueCurrency"), axis=1)
loan_df["AccruedCommission_USD"] = loan_df.apply(lambda x: to_usd(x, "AccruedCommission", "AccruedCommissionCurrency"), axis=1)

loan_months = sorted(loan_df["Month"].unique())
loan_accounts = sorted(loan_df["Account"].unique())

# get & preprocess data from SQL (Bond Custody Status)
query = "SELECT * FROM HST_MT535_D"
mt535_df = pd.read_sql_query(query, engine)
mt535_df["Pos_date"] = pd.to_datetime(mt535_df["Pos_date"], errors="coerce")
mt535_df["Holding"] = pd.to_numeric(mt535_df["Holding"], errors="coerce")
mt535_df["Loan"] = pd.to_numeric(mt535_df["Loan"], errors="coerce")
mt535_df["Collateral"] = pd.to_numeric(mt535_df["Collateral"], errors="coerce")
mt535_df["Month"] = mt535_df["Pos_date"].dt.to_period("M").astype(str)
mt535_df = mt535_df[mt535_df["Account"] != "21583"] # Filter out "21583" account

mt535_months = sorted(mt535_df["Month"].unique())

In [ ]:
# Plot 1 - Bond Custody Status (Stacked Bar + Line Chart)
df_mt535_clean = mt535_df.groupby(['Pos_date', 'Account']).agg({
    'Holding': 'sum',
    'Loan': 'sum',
    'Collateral': 'sum'
}).reset_index().sort_values('Pos_date')

df_mt535_total = df_mt535_clean.groupby('Pos_date').agg({
    'Holding': 'sum',
    'Collateral': 'sum'
}).reset_index()

fig_custody = go.Figure()

# --- A. SUM (Only shown when "ALL ACCOUNTS" is selected) ---
fig_custody.add_trace(go.Scatter(
    x=df_mt535_total['Pos_date'], y=df_mt535_total['Holding'],
    name='總持有 (Holding)', mode='lines+markers',
    line=dict(color='#1f77b4', width=3),
    customdata=['TOTAL_LINE'] * len(df_mt535_total),
    visible=True
))
fig_custody.add_trace(go.Scatter(
    x=df_mt535_total['Pos_date'], y=df_mt535_total['Collateral'],
    name='總擔保 (Collateral)', mode='lines',
    line=dict(color='#2ca02c', width=2, dash='dot'),
    customdata=['TOTAL_LINE'] * len(df_mt535_total),
    visible=True
))

# --- B. INDIVIDUAL (Each bar has has a bar + 2 lines, shown when a specific account is selected) ---
for acc in mt535_df["Account"].unique():
    acc_df = df_mt535_clean[df_mt535_clean['Account'] == acc]
    
    fig_custody.add_trace(go.Bar(
        x=acc_df['Pos_date'], y=acc_df['Loan'],
        name=f'借出-{acc}',
        customdata=[f'BAR_{acc}'] * len(acc_df),
        visible=True
    ))
    
    fig_custody.add_trace(go.Scatter(
        x=acc_df['Pos_date'], y=acc_df['Holding'],
        name=f'持有-{acc}', mode='lines+markers',
        line=dict(width=2),
        customdata=[f'LINE_{acc}'] * len(acc_df),
        visible=False
    ))
    
    fig_custody.add_trace(go.Scatter(
        x=acc_df['Pos_date'], y=acc_df['Collateral'],
        name=f'擔保-{acc}', mode='lines',
        line=dict(width=2, dash='dot'),
        customdata=[f'LINE_{acc}'] * len(acc_df),
        visible=False
    ))

fig_custody.update_layout(
    barmode='group',
    xaxis=dict(type='date', title='日期', tickformat='%Y-%m-%d'),
    yaxis=dict(title='金額 (USD)', autorange=True),
    hovermode='x unified',
    template='plotly_white',
    height=500
)

In [ ]:
# PLot 2 - 日借出量與累計利息走勢 (Bar/Scatter)
loan_months = sorted(loan_df["Month"].dropna().unique())
loan_accounts = sorted(loan_df["Account"].dropna().unique())

# ==========================================
# 1. 確保數字皆為數值型態，如果是 NaN 則補 0
# ==========================================
loan_df["LoanQuantity"] = pd.to_numeric(loan_df["LoanQuantity"], errors="coerce").fillna(0)
loan_df["LoanQuantity_USD"] = pd.to_numeric(loan_df["LoanQuantity_USD"], errors="coerce").fillna(0)
loan_df["AccruedCommission"] = pd.to_numeric(loan_df["AccruedCommission"], errors="coerce").fillna(0)
loan_df["AccruedCommission_USD"] = pd.to_numeric(loan_df["AccruedCommission_USD"], errors="coerce").fillna(0)

# ==========================================
# 2. 強制把同日期、同帳號、同 ISIN 的多筆合約數值「全部加總」
# ==========================================
daily_master = loan_df.groupby(
    ["Pos_date", "Month", "ISIN", "SecurityName", "Account", 
     "LoanValueCurrency", "AccruedCommissionCurrency"], 
    as_index=False
).agg({
    "LoanQuantity": "sum",
    "LoanQuantity_USD": "sum",
    "AccruedCommission": "sum",
    "AccruedCommission_USD": "sum"
}).sort_values(["ISIN", "Pos_date"])

# ==========================================
# 3. 分別取得加總後的借出量與累計利息
# ==========================================
daily_qty = daily_master[daily_master["LoanQuantity"] > 0]
daily_acc = daily_master[daily_master["AccruedCommission"] > 0]

isins = sorted(daily_qty["ISIN"].unique())
colors = px.colors.qualitative.Set2
isin_color_map = {isin: colors[i % len(colors)] for i, isin in enumerate(isins)}
# -----------------------------------------------------------------------------------
fig_loan = make_subplots(specs=[[{"secondary_y": True}]])

# ── Dummy trace ───────────────────────────────────────────────────────────────
for i, isin in enumerate(isins):
    name = (daily_qty[daily_qty["ISIN"] == isin]["SecurityName"].iloc[0]
            if not daily_qty[daily_qty["ISIN"] == isin].empty else isin)

    fig_loan.add_trace(go.Bar(
        x=[None], y=[None], name=name,
        marker_color=isin_color_map[isin],
        legendgroup="group_bar",
        legendgrouptitle=dict(text="借出量(原幣)"),
        showlegend=True,
        customdata=[["__dummy__", "__dummy__", "", "", isin]],  # ← index 4 加 ISIN
    ), secondary_y=False)

    fig_loan.add_trace(go.Scatter(
        x=[None], y=[None], mode="lines+markers", name=name,
        line=dict(color=isin_color_map[isin], dash="dot"),
        legendgroup="group_scatter",
        legendgrouptitle=dict(text="當月累計利息收入 (約當 USD)"),
        showlegend=True,
        customdata=[["__dummy__", "__dummy__", "", "", isin]],  # ← index 4 加 ISIN
    ), secondary_y=True)

# ── 實際資料 trace：customdata 也加入 ISIN ─────────────────────────────────────
for m in loan_months:
    for acc in loan_accounts:
        for isin in isins:
            qty_sub = daily_qty[
                (daily_qty["ISIN"] == isin) &
                (daily_qty["Month"] == m) &
                (daily_qty["Account"] == acc)
            ]
            if not qty_sub.empty:
                cd = list(zip([m]*len(qty_sub), [acc]*len(qty_sub),
                              qty_sub["LoanQuantity"], qty_sub["LoanValueCurrency"],
                              [isin]*len(qty_sub)))   # ← index 4 加 ISIN
                fig_loan.add_trace(go.Bar(
                    x=qty_sub["Pos_date"] + pd.Timedelta(hours=12),
                    y=qty_sub["LoanQuantity"],
                    name=qty_sub["SecurityName"].iloc[0],
                    marker_color=isin_color_map[isin],
                    legendgroup="group_bar",
                    showlegend=False,
                    customdata=cd,
                    hovertemplate="借出量: %{customdata[2]:,.2f} %{customdata[3]}<extra></extra>"
                ), secondary_y=False)

            acc_sub = daily_acc[
                (daily_acc["ISIN"] == isin) &
                (daily_acc["Month"] == m) &
                (daily_acc["Account"] == acc)
            ]
            if not acc_sub.empty:
                trace_currency = str(acc_sub["AccruedCommissionCurrency"].iloc[0]).upper()
                hover_temp = (
                    "<b>原幣利息: %{customdata[2]:,.2f} USD</b><extra></extra>"
                    if trace_currency == "USD" else
                    "<b>約當利息: %{y:,.2f} USD</b><br>原幣利息: %{customdata[2]:,.2f} %{customdata[3]}<extra></extra>"
                )
                cd = list(zip([m]*len(acc_sub), [acc]*len(acc_sub),
                              acc_sub["AccruedCommission"], acc_sub["AccruedCommissionCurrency"],
                              [isin]*len(acc_sub)))   # ← index 4 加 ISIN
                fig_loan.add_trace(go.Scatter(
                    x=acc_sub["Pos_date"] + pd.Timedelta(hours=12),
                    y=acc_sub["AccruedCommission_USD"],
                    mode="lines+markers",
                    name=acc_sub["SecurityName"].iloc[0],
                    line=dict(color=isin_color_map[isin], dash="dot"),
                    legendgroup="group_scatter",
                    showlegend=False,
                    customdata=cd,
                    hovertemplate=hover_temp
                ), secondary_y=True)
# for m in loan_months:
#     for acc in loan_accounts:
#         for isin in isins:
#             qty_sub = daily_qty[(daily_qty["ISIN"] == isin) & (daily_qty["Month"] == m) & (daily_qty["Account"] == acc)]
#             acc_sub = daily_acc[(daily_acc["ISIN"] == isin) & (daily_acc["Month"] == m) & (daily_acc["Account"] == acc)]

#             if not qty_sub.empty:
#                 is_first_bar = isin not in seen_isins_qty
#                 cd = list(zip([m]*len(qty_sub), [acc]*len(qty_sub), qty_sub["LoanQuantity"], qty_sub["LoanValueCurrency"]))
#                 fig_loan.add_trace(go.Bar(
#                     x=qty_sub["Pos_date"] + pd.Timedelta(hours=12), 
#                     y=qty_sub["LoanQuantity"], 
#                     name=f"{qty_sub['SecurityName'].iloc[0]}", marker_color=isin_color_map[isin],
#                     legendgroup="group_bar", legendgrouptitle=dict(text="借出量(原幣)"), showlegend=is_first_bar, customdata=cd,
#                     hovertemplate="借出量: %{customdata[2]:,.2f} %{customdata[3]}<extra></extra>"
#                 ), secondary_y=False)
#                 seen_isins_qty.add(isin)
            
#             if not acc_sub.empty:
#                 is_first_scatter = isin not in seen_isins_acc
#                 trace_acc_currency = str(acc_sub["AccruedCommissionCurrency"].iloc[0]).upper()
#                 hover_temp_scatter = "<b>原幣利息: %{customdata[2]:,.2f} USD</b><extra></extra>" if trace_acc_currency == "USD" else "<b>約當利息: %{y:,.2f} USD</b><br>原幣利息: %{customdata[2]:,.2f} %{customdata[3]}<extra></extra>"
#                 cd = list(zip([m]*len(acc_sub), [acc]*len(acc_sub), acc_sub["AccruedCommission"], acc_sub["AccruedCommissionCurrency"]))
                
#                 fig_loan.add_trace(go.Scatter(
#                     x=acc_sub["Pos_date"] + pd.Timedelta(hours=12), y = acc_sub["AccruedCommission_USD"], mode="lines+markers",
#                     name=f"{acc_sub['SecurityName'].iloc[0]}", line=dict(color=isin_color_map[isin], dash='dot'),
#                     legendgroup="group_scatter", legendgrouptitle=dict(text="當月累計利息收入 (約當 USD)"), showlegend=is_first_scatter,
#                     customdata=cd, hovertemplate=hover_temp_scatter
#                 ), secondary_y=True)
#                 seen_isins_acc.add(isin)

used_currencies = set(loan_df["LoanValueCurrency"].dropna().unique()).union(set(loan_df["AccruedCommissionCurrency"].dropna().unique()))
cross_rates = [f"{c}/USD={fx_rates[c]/fx_rates['USD']:.4f}" for c in used_currencies if c in fx_rates and c not in ("USD", "TWD")]
fx_note_str = "、".join(cross_rates) if cross_rates else "無外幣轉換"

fig_loan.update_layout(
    barmode="stack", hovermode="x unified", height=650, margin=dict(l=50, r=50, t=80, b=140),
    xaxis=dict(type='date', tickformat="%m-%d", dtick=86400000.0, tick0="2000-01-01 00:00", ticklabelmode="period", automargin=True),
    annotations=[dict(text=f"匯率基準: {fx_note_str} | 取得: {fx_fetch_time}", xref="paper", yref="paper", x=0.95, y=-0.1, showarrow=False, xanchor="right", yanchor="top", font=dict(size=12, color="#666666"))]
)

In [ ]:
# Plot 3 - 各檔債券利息收入明細 (HTML Table)
all_tables_html = ""
filter_loan_months = loan_months + ["全部"]
filter_loan_accounts = loan_accounts + ["所有帳號"]

for m in filter_loan_months:
    for acc in filter_loan_accounts:
        temp_loan_df = daily_master.copy()
        if acc != "所有帳號": 
            temp_loan_df = temp_loan_df[temp_loan_df["Account"] == acc]
        
        if m != "全部":
            temp_loan_df = temp_loan_df[temp_loan_df["Month"] == m]
            if not temp_loan_df.empty:
                # 這裡直接取加總過後的最後一天（已是各合約相加的結果）
                month_table = temp_loan_df.sort_values("Pos_date").groupby(
                    ["ISIN", "SecurityName", "Account", "AccruedCommissionCurrency"], as_index=False
                ).last()
            else:
                month_table = pd.DataFrame(columns=["ISIN", "SecurityName", "Account", "AccruedCommissionCurrency", "AccruedCommission", "AccruedCommission_USD"])
        else:
            if not temp_loan_df.empty:
                # 先取每個「月」最後一天的數值
                monthly_last = temp_loan_df.sort_values("Pos_date").groupby(
                    ["Month", "ISIN", "SecurityName", "Account", "AccruedCommissionCurrency"], as_index=False
                ).last()
                # 將「每個月」的數值相加
                month_table = monthly_last.groupby(
                    ["ISIN", "SecurityName", "Account", "AccruedCommissionCurrency"], as_index=False
                ).agg({"AccruedCommission": "sum", "AccruedCommission_USD": "sum"})
            else:
                month_table = pd.DataFrame(columns=["ISIN", "SecurityName", "Account", "AccruedCommissionCurrency", "AccruedCommission", "AccruedCommission_USD"])
        if not month_table.empty:
            month_table = month_table[["ISIN", "SecurityName", "Account", "AccruedCommissionCurrency", "AccruedCommission", "AccruedCommission_USD"]]
            month_table["AccruedCommission"] = month_table["AccruedCommission"].round(2)
            month_table["AccruedCommission_USD"] = month_table["AccruedCommission_USD"].round(2)
            month_table = month_table.sort_values("AccruedCommission_USD", ascending=False)
        
        header_usd = "利息加總 (約當 USD)" if m == "全部" else "當月累計利息 (約當 USD)"
        header_orig = "利息加總 (原幣)" if m == "全部" else "當月累計利息 (原幣)"
        
        table_html = month_table.to_html(index=False, classes='display cell-border', border=0)
        table_html = table_html.replace('<th>AccruedCommission_USD</th>', f'<th>{header_usd}</th>')\
                               .replace('<th>AccruedCommission</th>', f'<th>{header_orig}</th>')\
                               .replace('<th>AccruedCommissionCurrency</th>', '<th>利息原幣幣別</th>')\
                               .replace('<th>SecurityName</th>', '<th>債券名稱</th>')\
                               .replace('<th>Account</th>', '<th>保管帳戶</th>')
        total_commission = month_table["AccruedCommission_USD"].sum() if not month_table.empty else 0.0
        tfoot_html = f'''
                <tfoot>
                    <tr style="font-weight: bold; background-color: #f9f9f9;">
                        <td colspan="5" style="text-align: right; border-right: none;"></td>
                        <td style="text-align: right; border-left: none;">
                            <div style="display: flex; justify-content: space-between; align-items: center;">
                                <span>Total (USD)</span>
                                <span>{total_commission:,.2f}</span>
                            </div>
                        </td>
                    </tr>
                </tfoot>'''
        table_html = table_html.replace('</tbody>', f'</tbody>\n{tfoot_html}')
        
        display_style = "display: block;" if (m == "全部" and acc == "所有帳號") else "display: none;"
        all_tables_html += f"""<div class="table-container" data-month="{m}" data-account="{acc}" style="{display_style}"><h4 style="margin: 10px 0; color: #666;">資料範圍: {m} | 帳號: {acc}</h4>{table_html}</div>"""

In [ ]:
# HTML Layout
html_account_options = '<option value="所有帳號">所有帳號</option>' + "".join([f'<option value="{acc}">{acc}</option>' for acc in loan_accounts])
html_month_options = '<option value="全部">全部月份</option>' + "".join([f'<option value="{m}">{m}</option>' for m in loan_months])

html_mt535_acc_options = '<option value="所有帳號">所有帳號</option>' + "".join([f'<option value="{acc}">{acc}</option>' for acc in mt535_df["Account"].unique()])
html_mt535_month_options = '<option value="全部">全部月份</option>' + "".join([f'<option value="{m}">{m}</option>' for m in mt535_months])

html_template = f"""
<html>
<head>
    <meta charset="utf-8" />
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: sans-serif; margin: 30px; background-color: #f8f9fa; }}
        .section {{ background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 30px; }}
        .controls {{ display: flex; gap: 15px; margin-bottom: 15px; align-items: center; background: #f0f2f5; padding: 10px; border-radius: 4px; }}
        select {{ padding: 5px; border-radius: 4px; min-width: 150px; }}
        h2 {{ color: #333; border-left: 5px solid #007bff; padding-left: 10px; }}
        table.dataTable th:nth-child(5), table.dataTable td:nth-child(5), table.dataTable th:nth-child(6), table.dataTable td:nth-child(6) {{ text-align: right !important; padding-right: 20px !important; }}
    </style>
</head>
<body>

    <div class="section">
        <h2>1. 債券保管與借出狀態 (MT535 - 21582 & 88083)</h2>
        <div class="controls">
            <strong>篩選圖表：</strong>
            帳號 <select id="chart3-acc">{html_mt535_acc_options}</select>
            月份 <select id="chart3-month">{html_mt535_month_options}</select>
        </div>
        {fig_custody.to_html(full_html=False, include_plotlyjs='cdn', div_id="plotly-chart-3")}
    </div>

    <div class="section">
        <h2>2. 各月債券日借出量與累計利息走勢</h2>
        <div class="controls">
            <strong>篩選圖表：</strong>
            帳號 <select id="chart-acc">{html_account_options}</select>
            月份 <select id="chart-month">{html_month_options}</select>
        </div>
        {fig_loan.to_html(full_html=False, include_plotlyjs='cdn', div_id="plotly-chart")}
    </div>

    <div class="section">
        <h2>3. 各檔債券利息收入明細</h2>
        <div class="controls">
            <strong>篩選表格：</strong>
            帳號 <select id="table-acc">{html_account_options}</select>
            月份 <select id="table-month">{html_month_options}</select>
        </div>
        <div id="tables-wrapper">
            {all_tables_html}
        </div>
    </div>

    <script src="https://code.jquery.com/jquery-3.7.0.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
    $(document).ready(function() {{
        $('table').DataTable({{ paging: false, searching: false, info: false, order: [[5, 'desc']] }});

        function updateCustodyChart() {{
            var acc = $('#chart3-acc').val();
            var month = $('#chart3-month').val();
            var gd = document.getElementById('plotly-chart-3');
            if(!gd || !gd.data) return;

            var visibility = [];
            for (var i = 0; i < gd.data.length; i++) {{
                var tag = gd.data[i].customdata[0]; 
                
                if (acc === "所有帳號") {{
                    // 顯示總量線 (TOTAL_LINE) 和 所有的柱狀圖 (BAR_...)
                    visibility.push(tag === "TOTAL_LINE" || tag.startsWith("BAR_"));
                }} else {{
                    // 顯示該帳號的柱狀圖 (BAR_acc) 和 該帳號的明細線 (LINE_acc)
                    visibility.push(tag === "BAR_" + acc || tag === "LINE_" + acc);
                }}
            }}

            Plotly.restyle(gd, {{ visible: visibility }});

            // 月份篩選邏輯：使用 X 軸 range 縮放，確保折線圖不會因為斷點而消失
            var layoutUpdate = {{ 'yaxis.autorange': true }};
            if (month === "全部") {{
                layoutUpdate['xaxis.autorange'] = true;
            }} else {{
                var year = parseInt(month.split('-')[0]);
                var m = parseInt(month.split('-')[1]);
                var firstDay = month + "-01";
                var nextMonth = (m === 12) ? (year + 1) + "-01-01" : year + "-" + String(m + 1).padStart(2, '0') + "-01";
                
                layoutUpdate['xaxis.range'] = [firstDay, nextMonth];
                layoutUpdate['xaxis.autorange'] = false;
            }}
            Plotly.relayout(gd, layoutUpdate);
        }}

        /*
        function updateLoanChart() {{
            var acc = $('#chart-acc').val();
            var month = $('#chart-month').val();
            var gd = document.getElementById('plotly-chart');
            if(!gd || !gd.data) return;

            var visibility = [];
            for (var i = 0; i < gd.data.length; i++) {{
                var d = gd.data[i].customdata[0]; 
                visibility.push((month === "全部" || d[0] === month) && (acc === "所有帳號" || d[1] === acc));
            }}
            Plotly.restyle(gd, {{ visible: visibility }});

            var showLegendUpdates = [], firstVisibleLegendTrace = {{}};
            for (var i = 0; i < gd.data.length; i++) {{
                if (!visibility[i]) continue;
                var key = gd.data[i].name + '||' + (gd.data[i].legendgroup || '');
                if (!(key in firstVisibleLegendTrace)) firstVisibleLegendTrace[key] = i;
            }}
            for (var i = 0; i < gd.data.length; i++) {{
                var key = gd.data[i].name + '||' + (gd.data[i].legendgroup || '');
                showLegendUpdates.push(visibility[i] && firstVisibleLegendTrace[key] === i);
            }}
            Plotly.restyle(gd, {{ showlegend: [showLegendUpdates] }});
            Plotly.relayout(gd, {{ 'xaxis.type': 'date', 'yaxis.autorange': true, 'yaxis2.autorange': true }});
        }}
        */
        
        function updateLoanChart() {{
            var acc = $('#chart-acc').val();
            var month = $('#chart-month').val();
            var gd = document.getElementById('plotly-chart');
            if (!gd || !gd.data) return;

            // ── 第一階段：掃描實際資料 trace，統計哪些 ISIN 在目前篩選條件下有資料 ──
            var visibleIsins = new Set();
            for (var i = 0; i < gd.data.length; i++) {{
                var cd = gd.data[i].customdata;
                if (!cd || !cd[0] || cd[0][0] === "__dummy__") continue;  // 跳過 dummy
                var d = cd[0];
                var m_ok = (month === "全部" || d[0] === month);
                var a_ok = (acc === "所有帳號" || d[1] === acc);
                if (m_ok && a_ok) visibleIsins.add(d[4]);  // d[4] = ISIN
            }}

            // ── 第二階段：同時設定 visibility 與 showlegend ──────────────────────────
            var visibility = [], showlegend = [];
            for (var i = 0; i < gd.data.length; i++) {{
                var cd = gd.data[i].customdata;
                if (!cd || !cd[0] || cd[0][0] === "__dummy__") {{
                    var isin = cd[0][4];
                    visibility.push(true);                        // dummy 本身永遠存在
                    showlegend.push(visibleIsins.has(isin));      // 但只有資料存在時才顯示 legend
                }} else {{
                    var d = cd[0];
                    var vis = (month === "全部" || d[0] === month) &&
                            (acc === "所有帳號" || d[1] === acc);
                    visibility.push(vis);
                    showlegend.push(false);  // 實際資料 trace 永遠不顯示 legend
                }}
            }}

            Plotly.restyle(gd, {{ visible: visibility, showlegend: showlegend }});
            Plotly.relayout(gd, {{ 'xaxis.type': 'date', 'yaxis.autorange': true, 'yaxis2.autorange': true }});
        }}

        function updateTable() {{
            var acc = $('#table-acc').val();
            var month = $('#table-month').val();
            $('.table-container').hide();
            var target = $('.table-container[data-month="' + month + '"][data-account="' + acc + '"]');
            target.show();
            if ($.fn.dataTable.isDataTable(target.find('table'))) target.find('table').DataTable().columns.adjust();
        }}

        $('#chart3-acc, #chart3-month').on('change', updateCustodyChart);
        $('#chart-acc, #chart-month').on('change', updateLoanChart);
        $('#table-acc, #table-month').on('change', updateTable);
        setTimeout(updateCustodyChart, 600);
        setTimeout(updateLoanChart, 500);        
    }});
    </script>
</body>
</html>
"""

with open(filename, "w", encoding="utf-8") as f:
    f.write(html_template)

if not os.path.exists(target_folder):
    os.makedirs(target_folder)
target_path = os.path.join(target_folder, filename)
shutil.copy2(filename, target_path)

print(f"報表產生成功並儲存至：{target_path}")